<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/main/Preliminary_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
class Model():
  def __init__(self, name, model, answer_function):
    self.name = name
    self.model = model
    self.answer_function = answer_function

  def answer(self, question):
    return 0


In [3]:
!pip install -q -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 46.8 MB/s eta 0:00:00


In [7]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

In [8]:
from huggingface_hub import login
login(HF_TOKEN)

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [10]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype="auto",
    trust_remote_code=True,
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [11]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [12]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

In [13]:
messages = [
    {"role": "system", "content": "Sei un esperto di storia. Rispondi Ad una delle opzioni (A, B, C o D)"},
    {"role": "user", "content": "Chi è napoleone? A. Un politico, B: un personaggio televisivo C. STO CAZZO D. Un calciatore"},
]

In [14]:
generation_args = {
    "max_new_tokens": 600,
    "return_full_text": False,
    "temperature": 0.5,
    "do_sample": True,
}

In [16]:
output = pipe(messages, **generation_args)
print(output[0]['generated_text'])

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=600) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


La risposta corretta è A. Un politico.

Napoleone Bonaparte (1769-1821) fu un politico, un generale e un imperatore francese. È considerato uno dei personaggi più importanti della storia, soprattutto per la sua campagna militare che lo portò a diventare il più potente esponente politico d'Europa.


## Answer Extraction: Matching Model Output to Options

The LLM generates free-form text, but the game expects a single choice: **A, B, C, or D**.
We implement three strategies (in order of increasing sophistication) to map the model response back to one of the provided options:

1. **Regex extraction** – look for explicit letter mentions (fast & zero-cost)
2. **TF-IDF + Cosine Similarity** – Vector Space Model approach covered in the lectures
3. **Sentence-BERT (sBERT) Semantic Similarity** – deep embedding approach from the semantic search lectures

A combiner function tries each strategy in order, falling back to the next one if no confident match is found.

In [18]:
# Strategy 1: Regex-based direct extraction
import re

def extract_by_regex(model_output: str):
    text = model_output.strip()

    m = re.search(r'(?:answer(?:\s+is)?|option|choice|select|pick|correct)\s*[:\-]?\s*\*{0,2}([ABCD])\*{0,2}', text, re.IGNORECASE)
    if m: return m.group(1).upper()

    m = re.search(r'\b([ABCD])[\)\.:](?:\s|$)', text, re.IGNORECASE)
    if m: return m.group(1).upper()
    # lone capital letter (models often deliberate, then conclude with the letter)
    letters = re.findall(r'(?<![a-zA-Z])([ABCD])(?![a-zA-Z])', text)
    if letters:
        return letters[-1].upper()
    return None

test_cases = [
    ('The answer is A, because Napoleon was a political figure.', 'A'),
    ('Napoleon was a ruler. I think the answer is B.', 'B'),
    ('C) Un politico', 'C'),
    ('Napoleon era un generale. La risposta corretta e D.', 'D'),
    ('He was born in Corsica and rose to become emperor', None),
]
print('Regex extraction tests:')
for text, expected in test_cases:
    result = extract_by_regex(text)
    status = 'PASS' if result == expected else 'FAIL'
    print(f'  [{status}] Input: {repr(text[:55]):57s} -> Got: {result}, Expected: {expected}')

Regex extraction tests:
  [PASS] Input: 'The answer is A, because Napoleon was a political figur' -> Got: A, Expected: A
  [PASS] Input: 'Napoleon was a ruler. I think the answer is B.'          -> Got: B, Expected: B
  [PASS] Input: 'C) Un politico'                                          -> Got: C, Expected: C
  [PASS] Input: 'Napoleon era un generale. La risposta corretta e D.'     -> Got: D, Expected: D
  [PASS] Input: 'He was born in Corsica and rose to become emperor'       -> Got: None, Expected: None


In [19]:
!pip install -q scikit-learn

In [20]:
# Strategy 2: TF-IDF + Cosine Similarity (Vector Space Model)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def pick_by_tfidf(model_output: str, options: dict):

    labels = list(options.keys())
    texts = [model_output] + [options[l] for l in labels]

    # Fit TF-IDF on all texts together so the vocabulary is shared
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(texts)


    query_vec = tfidf_matrix[0]
    option_vecs = tfidf_matrix[1:]

    sims = cosine_similarity(query_vec, option_vecs)[0]
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best, scores

options_test = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'qualcosa di assurdo',
    'D': 'Un calciatore'
}
model_response = 'Napoleon era un grande leader politico e militare, imperatore dei francesi.'
best_tfidf, scores_tfidf = pick_by_tfidf(model_response, options_test)
print('TF-IDF cosine similarity scores:')
for label, score in sorted(scores_tfidf.items(), key=lambda x: -x[1]):
    marker = ' <-- SELECTED' if label == best_tfidf else ''
    print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
print(f'\nPicked option: {best_tfidf}')

TF-IDF cosine similarity scores:
  A ('Un politico'                 ): 0.3286 <-- SELECTED
  D ('Un calciatore'               ): 0.0923
  B ('un personaggio televisivo'   ): 0.0696
  C ('qualcosa di assurdo'         ): 0.0000

Picked option: A


In [21]:
!pip install -q sentence-transformers

In [22]:
# Strategy 3: Sentence-BERT (sBERT) Semantic Similarity

from sentence_transformers import SentenceTransformer
import numpy as np

sbert_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def pick_by_sbert(model_output: str, options: dict):

    labels = list(options.keys())
    all_texts = [model_output] + [options[l] for l in labels]

    embeddings = sbert_model.encode(all_texts, convert_to_numpy=True, normalize_embeddings=True)

    query_emb = embeddings[0]
    option_embs = embeddings[1:]

    # Dot product of unit vectors = cosine similarity
    sims = option_embs @ query_emb
    scores = {labels[i]: float(sims[i]) for i in range(len(labels))}
    best = max(scores, key=scores.get)
    return best, scores

best_sbert, scores_sbert = pick_by_sbert(model_response, options_test)
print('sBERT semantic similarity scores:')
for label, score in sorted(scores_sbert.items(), key=lambda x: -x[1]):
    marker = ' <-- SELECTED' if label == best_sbert else ''
    print(f'  {label} ({options_test[label]!r:30s}): {score:.4f}{marker}')
print(f'\nPicked option: {best_sbert}')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

sBERT semantic similarity scores:
  A ('Un politico'                 ): 0.3535 <-- SELECTED
  D ('Un calciatore'               ): 0.1437
  B ('un personaggio televisivo'   ): 0.0086
  C ('qualcosa di assurdo'         ): -0.1167

Picked option: A


In [23]:
# Combined Answer Selector -- the full pipeline

def select_option(model_output: str, options: dict, use_sbert: bool = True, verbose: bool = False) -> str:
    # --- Stage 1: Regex ---
    regex_pick = extract_by_regex(model_output)
    if regex_pick and regex_pick in options:
        if verbose:
            print(f'[Strategy 1 - Regex] Confident match found: {regex_pick}')
        return regex_pick
    if verbose:
        print(f'[Strategy 1 - Regex] No match (got {regex_pick!r}), falling back to similarity...')

    # --- Stage 2/3: Similarity ---
    if use_sbert:
        best, scores = pick_by_sbert(model_output, options)
        method = 'Strategy 3 - sBERT'
    else:
        best, scores = pick_by_tfidf(model_output, options)
        method = 'Strategy 2 - TF-IDF'

    if verbose:
        sorted_scores = sorted(scores.items(), key=lambda x: -x[1])
        print(f'[{method}] Similarity scores:')
        for label, score in sorted_scores:
            marker = ' <-- SELECTED' if label == best else ''
            print(f'  {label} ({options[label]!r:35s}): {score:.4f}{marker}')
        gap = sorted_scores[0][1] - sorted_scores[1][1] if len(sorted_scores) > 1 else 1.0
        confidence = 'HIGH' if gap > 0.10 else ('MEDIUM' if gap > 0.03 else 'LOW')
        print(f'  Confidence gap: {gap:.4f} ({confidence})')

    return best

example_options = {
    'A': 'Un politico',
    'B': 'un personaggio televisivo',
    'C': 'STO CAZZO',
    'D': 'Un calciatore'
}

raw_answer = output[0]['generated_text']
print('=' * 65)
print('Model raw output:')
print(raw_answer)
print('=' * 65)
final = select_option(raw_answer, example_options, use_sbert=True, verbose=True)
print('=' * 65)
print(f'FINAL ANSWER SUBMITTED TO GAME: {final}')

Model raw output:
La risposta corretta è A. Un politico.

Napoleone Bonaparte (1769-1821) fu un politico, un generale e un imperatore francese. È considerato uno dei personaggi più importanti della storia, soprattutto per la sua campagna militare che lo portò a diventare il più potente esponente politico d'Europa.
[Strategy 1 - Regex] Confident match found: A
FINAL ANSWER SUBMITTED TO GAME: A


## Summary: Three Strategies Compared

| Strategy | Method | Pros | Cons |
|---|---|---|---|
| **1. Regex** | Pattern matching | Zero cost, instant | Fails when model reasons without naming a letter |
| **2. TF-IDF + Cosine** | Sparse vector similarity (VSM lectures) | No extra model needed | Purely lexical — misses synonyms and paraphrases |
| **3. sBERT + Cosine** | Dense embedding similarity (semantic search lectures) | Captures meaning; cross-lingual | Needs a second model loaded in memory |

**Recommended default:** `select_option(..., use_sbert=True)`.

sBERT captures that *"leader politico"* ≈ *"Un politico"* even with no shared tokens — exactly the  
kind of semantic matching needed when the LLM explains its reasoning in full sentences rather than  
naming a letter explicitly. The multilingual model also handles Italian questions gracefully.